# qust 高阶用法

这本 notebook 重点讲 Polars 里没有或不常见的 qust 能力：

- 显式流式 `calc_stream`；
- `batch` 类批算子：sort、cut、pivot、violin_profile；
- `snapshot_by`：流式快照与回放；
- `partition_by`：有序分区释放；
- Python UDF batch；
- DataPool/plugin；
- 自定义 expression namespace。


In [17]:
import sys
sys.path.insert(0, "/root/otters/otters-py/python")

import os
import importlib

import qust as qs
qs = importlib.reload(qs)

from qust import col, pms
from qust._polars import pl
import qust.datasource as qds

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(14)

DATA_KLINE = "/root/qust-py/examples/data/data_kline2.parquet"
DATA_FUTURE = "/root/qust-py/examples/data/kline_data_all.parquet"
DATA_STOCK = "/root/qust-py/examples/data/stock_data_kline.parquet"

from qust import params_type as pt
from qust.expr import ExprNamespace, register_expr_namespace



## 1. 显式流式 `calc_stream`：输入 -> batch -> 状态 -> 输出

先把 stream 讲透。stream 不是“更快的 `calc_data`”，而是完全不同的执行语义：

- `calc_data`：假设这次输入就是完整数据；
- `calc_stream`：假设数据会分批到达，runtime 要保留历史状态；
- rolling/expanding/持仓/交易状态这类算子，都会因为“是否保留历史状态”而影响结果。

下面我们用 `close` 的 rolling mean 做一个小例子。输入 120 行，每 30 行一个 batch。表达式：

```python
col("close").mean().rolling(20).alias("ma20")
```

每个 batch 到来时，前一个 batch 的最后 19 行会影响下一个 batch 开头的 rolling 结果，这就是 stream 状态的价值。


In [ ]:
data = pl.read_parquet(DATA_KLINE).filter(pl.col("ticker") == "au").head(120)
source = qds.from_dataframe(data.select("ticker", "datetime", "close"), chunk_size=30)

stream_expr = col.with_cols(
    col("close").mean().rolling(20).alias("ma20")
).select("datetime", "close", "ma20")

stream_out = stream_expr.runtime().calc_stream(source)

print("输入一共 120 行，chunk_size=30，所以会分成 4 个 batch。")
print("输出仍然是 120 行，因为 rolling mean 是逐行输出。")
display(stream_out.head(35))
display(stream_out.tail(8))


### 1.1 手动看每个 batch 的输入和输出

为了避免 stream 变成抽象概念，下面用同一个 runtime 手动喂 4 个 batch。

重点看第 2 个 batch 的前几行：它的 `ma20` 会使用第 1 个 batch 尾部的数据，而不是只在第 2 个 batch 内重新开始。


In [19]:
manual_rt = stream_expr.runtime()
for batch_id, batch in enumerate(data.select("ticker", "datetime", "close").iter_slices(30), start=1):
    out = manual_rt.calc_data(batch)
    print(f"batch {batch_id}: input rows={batch.height}, output rows={out.height}")
    display(out.head(3))
    display(out.tail(3))


batch 1: input rows=30, output rows=30


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 00:01:00,390.119995,null
2022-07-02 00:02:00.500,390.140015,null
2022-07-02 00:03:00,390.200012,null


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 00:28:00,389.640015,389.572998
2022-07-02 00:29:00,389.579987,389.557997
2022-07-02 00:30:01.500,389.5,389.537997


batch 2: input rows=30, output rows=30


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 00:31:02,389.579987,389.524997
2022-07-02 00:32:00,389.619995,389.512997
2022-07-02 00:33:01,389.76001,389.520998


...

datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 01:01:00,388.940002,389.331006
2022-07-02 01:02:00,388.980011,389.309006
2022-07-02 01:03:00,389.100006,389.292006


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 01:28:06,389.040009,388.891002
2022-07-02 01:29:00,389.059998,388.896002
2022-07-02 01:30:00.500,389.26001,388.921002


batch 4: input rows=30, output rows=30


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 01:31:01,389.420013,388.946002
2022-07-02 01:32:01,389.399994,388.976003
2022-07-02 01:33:00.500,389.440002,389.016002


datetime,close,ma20
datetime[ms],f64,f64
2022-07-02 01:58:02,389.679993,389.694
2022-07-02 01:59:00.500,389.799988,389.712
2022-07-02 02:00:00,389.820007,389.735001


## 2. batch.sort：批内排序

`batch.sort(by=...)` 对当前 batch 做排序。在 `over(...)` 下会按组内排序。


In [20]:
small = pl.DataFrame({
    "label": ["a", "a", "b", "b", "c", "c"],
    "v": [3, 1, 4, 2, 6, 5],
    "x": [0.3, 0.1, 0.4, 0.2, 0.6, 0.5],
})

col.all.batch.sort("v", descending=True).calc_data(small)


label,v,x
str,i64,f64
"""c""",6,0.6
"""c""",5,0.5
"""b""",4,0.4
"""a""",3,0.3
"""b""",2,0.2
"""a""",1,0.1


## 3. batch.cut：分箱

`batch.cut` 的 breaks 使用 `qust.params_type`：

- `pt.value(4)`：按值域生成 4 个箱；
- `pt.value([...])`：按给定边界分箱，缺 `-inf/inf` 时自动补；
- `pt.quantile(4)`：按分位数分箱；
- `labels=None` 时输出 struct，表示左右边界；否则输出 labels。


In [21]:
cut_struct = col("x").batch.cut(pt.value(3), labels=None).calc_data(small)
cut_label = col("x").batch.cut(pt.value([0.2, 0.4]), labels=["low", "mid", "high"]).calc_data(small)

display(cut_struct)
display(cut_label)


x
struct[2]
"{0.266667,0.433333}"
"{-inf,0.266667}"
"{0.266667,0.433333}"
"{-inf,0.266667}"
"{0.433333,inf}"
"{0.433333,inf}"


x
str
"""mid"""
"""low"""
"""mid"""
"""low"""
"""high"""
"""high"""


## 4. batch.pivot：固定输出 schema 的 pivot

qust 的执行图需要在规划阶段知道输出 schema，所以 `batch.pivot` 必须传 `on_columns`。

底层调用 `pl.DataFrame.pivot(...)`，参数基本保持一致。


In [22]:
pivot_out = col.all.batch.pivot(
    on="label",
    on_columns=["a", "b", "c"],
    values="v",
    aggregate_function="sum",
).calc_data(small)

pivot_out


x,a,b,c
f64,i64,i64,i64
0.3,3,0,0
0.1,1,0,0
0.4,0,4,0
0.2,0,2,0
0.6,0,0,6
0.5,0,0,5


## 5. violin_profile：先聚合分布，再交给 monitor.violin

`batch.violin_profile()` 会把原始样本压缩成绘图 profile。这样拖动/缩放时 monitor 不需要从原始样本重算 KDE。


In [23]:
profile_data = pl.DataFrame({
    "bucket": [1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 3],
    "ret": [0.01, 0.03, -0.02, 0.02, -0.01, 0.04, 0.05, -0.03, 0.01, 0.02, 0.06, 0.07],
})

profile = (
    col("ret")
    .batch.violin_profile(lower_bound=0.05, up_bound=0.95)
    .group_by("bucket")
    .batch.sort("bucket")
    .calc_data(profile_data)
)

profile.head(12)


bucket,count,min,q1,median,q3,max,y,half_width,is_point
i64,u64,f64,f64,f64,f64,f64,list[f64],list[f64],bool
1,2,0.01,0.0125,0.015,0.0175,0.02,"[0.01, 0.010101, … 0.02]","[0.881266, 0.886167, … 0.881266]",false
2,1,0.04,0.04,0.04,0.04,0.04,[],[],true
3,3,0.01,0.015,0.02,0.04,0.06,"[0.01, 0.010505, … 0.06]","[0.946318, 0.952663, … 0.595693]",false



## 6. snapshot_by：标签会回流时，为什么普通 stream 不够

普通 stream 很适合“时间一直往前走”的数据。但有些数据的标签会回流：

```text
batch1: a, a
batch2: b, a
batch3: b, c
```

如果我们按 `label` 计算累计和，`a` 在 batch2 又出现了。为了保证 `a` 的结果一致，`snapshot_by("label")` 会保存标签快照；当标签回流时，取出相关历史重新计算。

这类逻辑常见于：

- 分品种/分日期的数据不是严格连续到达；
- 数据修正、回放、异步到达；
- 标签维度未来还可能出现新行。


In [24]:
snap_data = pl.DataFrame({
    "label": ["a", "a", "b", "a", "b", "c"],
    "v": [1, 2, 10, 3, 20, 100],
})

snap_source = qds.from_dataframe(snap_data, chunk_size=2)
snap_expr = col("v").sum().expanding().snapshot_by("label", keep_labels=4).alias("cum_v")
snap_expr.runtime().calc_stream(snap_source)


cum_v
i64
1
3
1
3
10
6
1
3
10



## 7. partition_by：已经排序的数据，什么时候可以释放一个分区

`partition_by` 和 `snapshot_by` 不同，它假设输入已经按 key 排好序：

```text
a, a, a, b, b, c
```

规则是：最后一个 key 永远视为“还没结束”。只有看到新 key，前一个 key 才算完成，可以释放给下游。

例子：

- 收到 `a, a, a`：还没看到新 key，`a` 可能还没结束，不输出；
- 后面收到 `b, b`：看到 b 以后，a 完成，释放 a；
- 后面收到 `c`：看到 c 以后，b 完成，释放 b；
- c 是最后一个分区，没有后续 key 证明它完成，所以默认不 flush。

它适合已经按 `ticker/date` 排序的离线流，不适合乱序数据。


In [25]:
part_data = pl.DataFrame({
    "label": ["a", "a", "a", "b", "b", "c"],
    "v": [1, 2, 3, 4, 5, 6],
})

part_source = qds.from_dataframe(part_data, chunk_size=3)
part_expr = col.all.partition_by(col("label"), prompt_partition=False)
part_expr.runtime().calc_stream(part_source)


label,v
str,i64
"""a""",1
"""a""",2
"""a""",3
"""b""",4
"""b""",5


## 8. Python UDF batch

当某个算子暂时没有 Rust 原生实现时，可以用 `qs.UdfBatch`。UDF 接收 `Packet`，通过 `packet.get_dataframe(None)` 取当前输入 batch。

原则：能用现有 qust 算子组合就先组合；只有核心逻辑不能组合时再写 UDF。


In [26]:
class AddBatchSummary(qs.UdfBatch):
    def calc_batch(self, packet):
        df = packet.get_dataframe(None)
        return df.with_columns(
            pl.col("v").mean().alias("batch_mean_v"),
            pl.len().alias("batch_rows"),
        )

udf_out = col("label", "v").udf.batch(AddBatchSummary()).calc_data(part_data)
udf_out


label,v,batch_mean_v,batch_rows
str,i64,f64,u32
"""a""",1,3.5,6
"""a""",2,3.5,6
"""a""",3,3.5,6
"""b""",4,3.5,6
"""b""",5,3.5,6
"""c""",6,3.5,6


## 9. DataPool/plugin：保存中间结果

`save_data_inner(key, pool_key)` 会把当前表达式输出保存到 runtime 的 `DataPool`，表达式本身继续往下执行。

这个适合 debug、monitor callback、或者把中间表拿出来做额外检查。


In [27]:
runtime = col(
    col("label", "v").save_data_inner("raw_input", "demo_pool"),
    (col("v") * col.lit(10)).alias("v10"),
).runtime()

result = runtime.calc_data(part_data)
saved = qs.DataPool("demo_pool").get_plugin(runtime).get_dataframe("raw_input")

display(result)
display(saved)


label,v,v10
str,i64,f64
"""a""",1,10.0
"""a""",2,20.0
"""a""",3,30.0
"""b""",4,40.0
"""b""",5,50.0
"""c""",6,60.0


label,v
str,i64
"""a""",1
"""a""",2
"""a""",3
"""b""",4
"""b""",5
"""c""",6


## 10. 自定义 namespace

可以用 `register_expr_namespace("name")` 给 `Expr` 扩展命名空间。下面的 `demo_ops.ret()` 只是组合现有 qust 算子，不写新的执行内核。


In [28]:
@register_expr_namespace("demo_ops")
class DemoOps(ExprNamespace):
    def ret(self):
        return self._e / self._e.shift(1) - col.lit(1.0)

col("close").demo_ops.ret().alias("ret").calc_data(data.head(8))


ret
f64
0.000616
0.000667
0.000821
0.000718
0.000564
0.000667
0.0
0.0
